In [ ]:
import pandas as pd
import numpy as np
from functions import scrape_nba_data
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [ ]:
raw_df = scrape_nba_data(2024, 2026)
raw_df.head()

In [ ]:
raw_df.describe()

In [ ]:
raw_df.dtypes

In [ ]:
# Some players get traded mid-season: the API gives one row per team plus a
# "TOT" (total) row combining them. Keep only the TOT row when present so
# each player has exactly one row per season. (A groupby(...).apply() here
# would silently drop PLAYER_ID/SEASON under current pandas, since group-by
# columns are excluded from what's passed to the function — so this uses a
# vectorized transform instead.)
has_tot = raw_df.groupby(["PLAYER_ID", "SEASON"])["TEAM_ABBREVIATION"].transform(
    lambda s: (s == "TOT").any()
)
df = raw_df[~has_tot | (raw_df["TEAM_ABBREVIATION"] == "TOT")].reset_index(drop=True)

# eda_box_scores.py expects the scraper's original column names (SEASON,
# PLAYER_ID, FG_PCT, GP, MIN, FGA, ...), so the deduplicated data is saved
# under those names — this is the file the EDA cell below reads.
df.to_csv("nba_data.csv", index=False)

# A friendlier, renamed per-game table for display/export. Not used by the
# EDA script, which relies on the original column names saved above.
id_cols = ["SEASON", "PLAYER_NAME", "TEAM_ABBREVIATION", "AGE", "PLAYER_POSITION"]

shooting_cols = ["FG_PCT", "FG3_PCT", "FT_PCT"]

per_game_cols = ["PTS", "REB", "AST", "STL", "TOV"]

advanced_cols = [
    "OFF_RATING", "DEF_RATING", "NET_RATING",
    "TS_PCT", "EFG_PCT", "USG_PCT", "AST_PCT", "REB_PCT", "PIE", "PACE",
]

keep_cols = id_cols + shooting_cols + per_game_cols + advanced_cols
keep_cols = [c for c in keep_cols if c in df.columns]

per_game = df[keep_cols].rename(columns={
    "SEASON": "YEAR",
    "PLAYER_NAME": "PLAYER",
    "TEAM_ABBREVIATION": "TEAM",
    "PTS": "PPG",
    "REB": "RPG",
    "AST": "APG",
    "STL": "SPG",
    "TOV": "TOPG",
})

per_game = per_game.sort_values(["YEAR", "PLAYER"]).reset_index(drop=True)

per_game.to_csv("nba_per_game.csv", index=False)
per_game.head()


In [ ]:
from eda_box_scores import run_eda

run_eda(input_csv="nba_data.csv", output_dir="eda_output")